# 8 · Interaction & dashboards

The payoff of a web map is interaction. `hover` customises tooltips; `tap_profile`/`on_tap` respond to
clicks; `draw`/`aoi_crop` let the user draw an area of interest and crop a raster to it; `cross_filter`
links selections across panels. `quickmap(backend="interactive")` is the one-call entry point, and the
dashboard builders (`dashboard`, `layer_control`, `attribute_table`) assemble Panel apps. Many of these are
**live** — they need a running kernel/server to fire callbacks — so here we build them and show the result.

**Setup** — Bokeh extension and the Rhine gauges + Lisbon DEM.

In [ ]:
from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import holoviews as hv
hv.extension("bokeh")            # the interactive tier renders through Bokeh

from pyramids.dataset import Dataset
from pyramids.feature import FeatureCollection
from digitalearth.interactive import InteractiveMap

gauges = FeatureCollection.read_file(str(DATA / "rhine_gauges.geojson"))
dem = Dataset.read_file(str(DATA / "LisbonElevation.tif"))

### `quickmap(backend='interactive')` — one call
The same `quickmap` entry point as the static tier; `backend='interactive'` returns an `InteractiveMap` fed from the same auto-style pipeline.

In [ ]:
from digitalearth import quickmap

m = quickmap(gauges, crs=3857, backend="interactive", basemap=True)
m

### `hover` — custom tooltips
Enable hover with a chosen tooltip list. Move the cursor over a gauge to see its name and discharge.

In [ ]:
m = InteractiveMap(crs=3857, title="hover the gauges")
m.points(gauges, value_column="discharge", size=10, cmap="plasma")
m.hover(tooltips=[("name", "@name"), ("discharge", "@discharge")])
m

### `draw` + `aoi_crop` — draw an area of interest
`draw('box')` adds a box-draw tool; in a live kernel you draw a rectangle and `aoi_crop(dataset)` clips a pyramids raster to it. Here we add the tool over the DEM (draw a box with the toolbar's box-edit tool).

In [ ]:
m = InteractiveMap(crs=dem.epsg, title="draw a box to crop")
m.image(dem, cmap="terrain").draw("box")
m

### `layer_control` — a layer panel
`layer_control` wraps the map in a Panel app with opacity sliders and a basemap switch — a mini map application.

In [ ]:
m = InteractiveMap(crs=3857, title="layer control")
m.tiles("CartoLight").points(gauges, value_column="discharge", size=9, cmap="plasma")
m.layer_control()

### `attribute_table` — linked table
Show the gauge attributes as a Panel table linked to the map (selecting rows highlights features in a live session).

In [ ]:
m = InteractiveMap(crs=3857)
m.points(gauges, value_column="discharge", size=8, cmap="plasma")
m.attribute_table(gauges)